## Cell 1: Install Dependencies

In [1]:
!pip install transformers torch torchvision Pillow datasets seqeval scikit-learn


Defaulting to user installation because normal site-packages is not writeable


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2: Imports

In [2]:
import os
import json
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

print("All imports successful")

All imports successful


## Cell 3: Configuration

In [3]:
MODEL_NAME    = "microsoft/layoutlmv3-base"
DATASET_PATH  = "C:/Users/PRIYANKA/Desktop/Cura/datasets/training_images"
NUM_CLASSES   = 3
CLASS_NAMES   = ["non_medical", "prescription", "report"]
EPOCHS        = 15
BATCH_SIZE    = 4
LEARNING_RATE = 2e-5
MAX_LENGTH    = 512
IMAGE_SIZE    = 224

SAVE_PATH = "saved_models/classifier/layoutlmv3-classifier"
os.makedirs(SAVE_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Model      : {MODEL_NAME}")
print(f"Dataset    : {DATASET_PATH}")
print(f"Classes    : {CLASS_NAMES}")
print(f"Device     : {device}")
print(f"Save path  : {SAVE_PATH}")

Model      : microsoft/layoutlmv3-base
Dataset    : C:/Users/PRIYANKA/Desktop/Cura/datasets/training_images
Classes    : ['non_medical', 'prescription', 'report']
Device     : cuda
Save path  : saved_models/classifier/layoutlmv3-classifier


## Cell 4: PaddleOCR â€” Text + Bounding Box Extraction

LayoutLMv3 requires words **and** bounding boxes normalized to the 0â€“1000 range.
We use PaddleOCR (`apply_ocr=False` on the processor) instead of the default Tesseract.

In [4]:
import sys
import json
sys.path.insert(0, "C:/Users/PRIYANKA/Desktop/Cura")

OCR_CACHE_FILE = "C:/Users/PRIYANKA/Desktop/Cura/datasets/ocr_cache.json"

ocr_cache = {}
_live_ocr = None  # initialized lazily only if a cache miss occurs

if Path(OCR_CACHE_FILE).exists():
    with open(OCR_CACHE_FILE, encoding="utf-8") as f:
        ocr_cache = json.load(f)
    print(f"OCR cache loaded : {len(ocr_cache)} entries")
    print("No live OCR will run during training.")
else:
    print("WARNING: ocr_cache.json not found.")
    print("Run `python scripts/precompute_ocr.py` first.")


def get_ocr_data(image_path):
    global _live_ocr
    key = str(Path(image_path).resolve())
    if key in ocr_cache:
        entry = ocr_cache[key]
        return entry["words"], entry["boxes"]
    if _live_ocr is None:
        from models.ocr_engine import OCREngine
        print("Cache miss â€” initializing live OCR engine ...")
        _live_ocr = OCREngine()
    print(f"Cache miss: {Path(image_path).name}")
    return _live_ocr.extract_ocr_data(image_path)


print("get_ocr_data() ready.")


OCR cache loaded : 780 entries
No live OCR will run during training.
get_ocr_data() ready.


## Cell 5: Load LayoutLMv3 Processor

`apply_ocr=False` tells the processor to accept our PaddleOCR output rather than running Tesseract.

In [5]:
processor = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)
print(f"Processor loaded : {MODEL_NAME}")
print("apply_ocr        : False  (using PaddleOCR)")

Processor loaded : microsoft/layoutlmv3-base
apply_ocr        : False  (using PaddleOCR)


## Cell 6: Custom PyTorch Dataset

In [6]:
class DocumentDataset(Dataset):
    def __init__(self, image_paths, labels, processor, max_length=512):
        self.image_paths = image_paths
        self.labels      = labels
        self.processor   = processor
        self.max_length  = max_length

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label      = self.labels[idx]

        image        = Image.open(image_path).convert("RGB")
        words, boxes = get_ocr_data(image_path)  # reads from cache

        encoding = self.processor(
            images=image,
            text=words,
            boxes=boxes,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "bbox":           encoding["bbox"].squeeze(0),
            "pixel_values":   encoding["pixel_values"].squeeze(0),
            "label":          torch.tensor(label, dtype=torch.long),
        }


print("DocumentDataset class defined.")


DocumentDataset class defined.


## Cell 7: Load Dataset â€” Group-Aware Split (70 / 15 / 15, stratified)

All 6 augmented variants of the same source image are kept in the same split
to prevent data leakage. Splitting is done at source-group level, then
stratified by class so each split has a balanced class distribution.

In [7]:
import re
from sklearn.model_selection import GroupShuffleSplit

label_map = {name: idx for idx, name in enumerate(CLASS_NAMES)}
all_paths, all_labels, all_groups = [], [], []

for class_name, label in label_map.items():
    class_dir = Path(DATASET_PATH) / class_name
    if not class_dir.exists():
        print(f"WARNING: {class_dir} not found â€” skipping")
        continue
    images = sorted([
        str(p) for p in class_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
    ])
    for img_path in images:
        stem  = Path(img_path).stem
        group = re.sub(r'_(b|br|d|r|c)$', '', stem)  # strip augmentation suffix
        all_paths.append(img_path)
        all_labels.append(label)
        all_groups.append(group)
    print(f"  {class_name:15s}: {len(images):4d} images  (label={label})")

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
all_groups = np.array(all_groups)

print(f"\nTotal images  : {len(all_paths)}")
print(f"Unique groups : {len(set(all_groups))}")

# First split: 70% train, 30% temp
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(all_paths, all_labels, all_groups))

# Second split: temp 50/50 â†’ val (15%) and test (15%)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(
    all_paths[temp_idx], all_labels[temp_idx], all_groups[temp_idx]
))
val_idx  = temp_idx[val_idx]
test_idx = temp_idx[test_idx]

train_paths  = all_paths[train_idx].tolist()
train_labels = all_labels[train_idx].tolist()
val_paths    = all_paths[val_idx].tolist()
val_labels   = all_labels[val_idx].tolist()
test_paths   = all_paths[test_idx].tolist()
test_labels  = all_labels[test_idx].tolist()

for split_name, paths, labels in [
    ("Train", train_paths, train_labels),
    ("Val",   val_paths,   val_labels),
    ("Test",  test_paths,  test_labels),
]:
    counts = {CLASS_NAMES[i]: labels.count(i) for i in range(NUM_CLASSES)}
    n_groups = len(set(re.sub(r'_(b|br|d|r|c)$', '', Path(p).stem) for p in paths))
    print(f"{split_name:6s} ({len(paths):4d} images, {n_groups:3d} groups): {counts}")


  non_medical    :  240 images  (label=0)
  prescription   :  300 images  (label=1)
  report         :  240 images  (label=2)

Total images  : 780
Unique groups : 130
Train  ( 546 images,  91 groups): {'non_medical': 162, 'prescription': 198, 'report': 186}
Val    ( 114 images,  19 groups): {'non_medical': 36, 'prescription': 48, 'report': 30}
Test   ( 120 images,  20 groups): {'non_medical': 42, 'prescription': 54, 'report': 24}


## Cell 8: DataLoaders

In [8]:
def collate_fn(batch):
    return {
        "input_ids":      torch.stack([b["input_ids"]      for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "bbox":           torch.stack([b["bbox"]           for b in batch]),
        "pixel_values":   torch.stack([b["pixel_values"]   for b in batch]),
        "label":          torch.stack([b["label"]          for b in batch]),
    }


train_dataset = DocumentDataset(train_paths, train_labels, processor, MAX_LENGTH)
val_dataset   = DocumentDataset(val_paths,   val_labels,   processor, MAX_LENGTH)
test_dataset  = DocumentDataset(test_paths,  test_labels,  processor, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

Train batches : 137
Val   batches : 29
Test  batches : 30


## Cell 9: Load LayoutLMv3 Model

In [9]:
model = LayoutLMv3ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label={0: "non_medical", 1: "prescription", 2: "report"},
    label2id={"non_medical": 0, "prescription": 1, "report": 2},
)
model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")
print(f"Device               : {device}")

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForSequenceClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                        | Status  | 
---------------------------+---------+-
classifier.dense.bias      | MISSING | 
classifier.dense.weight    | MISSING | 
classifier.out_proj.bias   | MISSING | 
classifier.out_proj.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters     : 125,919,875
Trainable parameters : 125,919,875
Device               : cuda


## Cell 10: Optimizer, Scheduler, Class Weights, and Training Bookkeeping

In [10]:
from sklearn.utils.class_weight import compute_class_weight

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.10 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# Weighted loss â€” prescription (300 imgs) vs non_medical/report (240 each)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_labels,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f"Class weights : { {CLASS_NAMES[i]: round(class_weights[i], 4) for i in range(NUM_CLASSES)} }")

history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}

best_val_f1      = 0.0
best_ckpt_path   = os.path.join(SAVE_PATH, "best_checkpoint")
patience         = 5
patience_counter = 0

print(f"Total training steps : {total_steps}")
print(f"Warmup steps         : {warmup_steps}")
print(f"Early stopping       : patience={patience}")


Class weights : {'non_medical': 1.1235, 'prescription': 0.9192, 'report': 0.9785}
Total training steps : 2055
Warmup steps         : 205
Early stopping       : patience=5


## Cell 11: Training Loop

In [11]:
from tqdm.auto import tqdm

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_train_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        bbox           = batch["bbox"].to(device)
        pixel_values   = batch["pixel_values"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox,
            pixel_values=pixel_values,
            labels=labels,
        )
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_train_loss += outputs.loss.item()

    avg_train_loss = epoch_train_loss / len(train_loader)

    # Validate
    model.eval()
    epoch_val_loss = 0.0
    all_preds, all_true = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            bbox           = batch["bbox"].to(device)
            pixel_values   = batch["pixel_values"].to(device)
            labels         = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                bbox=bbox,
                pixel_values=pixel_values,
                labels=labels,
            )
            epoch_val_loss += outputs.loss.item()
            all_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            all_true.extend(labels.cpu().numpy())

    avg_val_loss = epoch_val_loss / len(val_loader)
    val_acc      = accuracy_score(all_true, all_preds)
    val_f1       = f1_score(all_true, all_preds, average="weighted")

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(f"\nEpoch {epoch+1:02d}/{EPOCHS}  |  "
          f"Train Loss: {avg_train_loss:.4f}  |  "
          f"Val Loss: {avg_val_loss:.4f}  |  "
          f"Val Acc: {val_acc:.4f}  |  "
          f"Val F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained(best_ckpt_path)
        processor.save_pretrained(best_ckpt_path)
        patience_counter = 0
        print(f"  *** Best model saved (F1={best_val_f1:.4f}) ***")
    else:
        patience_counter += 1
        print(f"  No improvement â€” patience {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nTraining complete.  Best Val F1: {best_val_f1:.4f}")

Epoch 1/15 [Train]:   0%|          | 0/137 [00:00<?, ?it/s]

Epoch 1/15 [Val]:   0%|          | 0/29 [00:00<?, ?it/s]


Epoch 01/15  |  Train Loss: 1.0537  |  Val Loss: 0.8274  |  Val Acc: 0.6842  |  Val F1: 0.6634


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  *** Best model saved (F1=0.6634) ***


...

Epoch 7/15 [Val]:   0%|          | 0/29 [00:00<?, ?it/s]


Epoch 07/15  |  Train Loss: 0.0004  |  Val Loss: 0.0002  |  Val Acc: 1.0000  |  Val F1: 1.0000
  No improvement â€” patience 4/5


Epoch 8/15 [Train]:   0%|          | 0/137 [00:00<?, ?it/s]

Epoch 8/15 [Val]:   0%|          | 0/29 [00:00<?, ?it/s]


Epoch 08/15  |  Train Loss: 0.0004  |  Val Loss: 0.0002  |  Val Acc: 1.0000  |  Val F1: 1.0000
  No improvement â€” patience 5/5

Early stopping at epoch 8

Training complete.  Best Val F1: 1.0000


## Cell 12: Evaluate on Test Set

In [12]:
best_model = LayoutLMv3ForSequenceClassification.from_pretrained(best_ckpt_path)
best_model.to(device)
best_model.eval()

all_preds, all_true = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test evaluation"):
        outputs = best_model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            bbox=batch["bbox"].to(device),
            pixel_values=batch["pixel_values"].to(device),
        )
        all_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
        all_true.extend(batch["label"].numpy())

test_accuracy = accuracy_score(all_true, all_preds)
test_f1       = f1_score(all_true, all_preds, average="weighted")

print(f"\n=== Test Results ===")
print(f"Accuracy    : {test_accuracy:.4f}")
print(f"Weighted F1 : {test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES))
print("Confusion Matrix:")
print(confusion_matrix(all_true, all_preds))
print(f"Label order : {CLASS_NAMES}")

Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

Test evaluation:   0%|          | 0/30 [00:00<?, ?it/s]


=== Test Results ===
Accuracy    : 0.9083
Weighted F1 : 0.9068

Classification Report:
              precision    recall  f1-score   support

 non_medical       1.00      0.79      0.88        42
prescription       0.87      0.96      0.91        54
      report       0.89      1.00      0.94        24

    accuracy                           0.91       120
   macro avg       0.92      0.92      0.91       120
weighted avg       0.92      0.91      0.91       120

Confusion Matrix:
[[33  8  1]
 [ 0 52  2]
 [ 0  0 24]]
Label order : ['non_medical', 'prescription', 'report']


## Cell 13: Sample Predictions (1 Image per Class)

In [13]:
best_model.eval()

sample_by_class = {}
for path, lbl in zip(test_paths, test_labels):
    if lbl not in sample_by_class:
        sample_by_class[lbl] = path
    if len(sample_by_class) == NUM_CLASSES:
        break

print("=== Sample Predictions ===\n")

for lbl_idx in sorted(sample_by_class):
    img_path   = sample_by_class[lbl_idx]
    true_class = CLASS_NAMES[lbl_idx]

    image        = Image.open(img_path).convert("RGB")
    words, boxes = get_ocr_data(img_path)  # reads from cache

    enc = processor(
        images=image,
        text=words,
        boxes=boxes,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    with torch.no_grad():
        outputs = best_model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
            bbox=enc["bbox"].to(device),
            pixel_values=enc["pixel_values"].to(device),
        )

    probs      = F.softmax(outputs.logits, dim=-1).squeeze(0)
    pred_idx   = probs.argmax().item()
    pred_class = CLASS_NAMES[pred_idx]
    confidence = probs[pred_idx].item() * 100

    print(f"Image      : {Path(img_path).name}")
    print(f"True class : {true_class}")
    print(f"Predicted  : {pred_class}  ({confidence:.1f}%)")
    all_p = "  ".join(f"{CLASS_NAMES[i]}: {probs[i].item()*100:.1f}%" for i in range(NUM_CLASSES))
    print(f"All probs  : {all_p}")
    print()


=== Sample Predictions ===

Image      : non_medical_001.png
True class : non_medical
Predicted  : non_medical  (99.9%)
All probs  : non_medical: 99.9%  prescription: 0.0%  report: 0.0%

Image      : prescription_001.png
True class : prescription
Predicted  : prescription  (99.9%)
All probs  : non_medical: 0.0%  prescription: 99.9%  report: 0.0%

Image      : report_004.png
True class : report
Predicted  : report  (99.8%)
All probs  : non_medical: 0.0%  prescription: 0.2%  report: 99.8%



## Cell 14: Save Final Model + Processor + model_info.json

In [14]:
best_model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

model_info = {
    "model_architecture": "LayoutLMv3ForSequenceClassification",
    "class_names":        CLASS_NAMES,
    "num_classes":        NUM_CLASSES,
    "test_accuracy":      round(test_accuracy, 4),
    "test_f1":            round(test_f1, 4),
    "base_model":         MODEL_NAME,
}

info_path = os.path.join(SAVE_PATH, "model_info.json")
with open(info_path, "w") as f:
    json.dump(model_info, f, indent=2)

print(f"Model saved to  : {SAVE_PATH}")
print(f"model_info.json : {info_path}")
print(json.dumps(model_info, indent=2))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to  : saved_models/classifier/layoutlmv3-classifier
model_info.json : saved_models/classifier/layoutlmv3-classifier\model_info.json
{
  "model_architecture": "LayoutLMv3ForSequenceClassification",
  "class_names": [
    "non_medical",
    "prescription",
    "report"
  ],
  "num_classes": 3,
  "test_accuracy": 0.9083,
  "test_f1": 0.9068,
  "base_model": "microsoft/layoutlmv3-base"
}


## Cell 15: Verify Saved Model Loads Correctly

In [15]:
verify_processor = LayoutLMv3Processor.from_pretrained(SAVE_PATH, apply_ocr=False)
verify_model     = LayoutLMv3ForSequenceClassification.from_pretrained(SAVE_PATH)
verify_model.to(device)
verify_model.eval()

with open(os.path.join(SAVE_PATH, "model_info.json")) as f:
    loaded_info = json.load(f)

print(f"Loaded class names : {loaded_info['class_names']}")
print(f"Architecture       : {loaded_info['model_architecture']}")
print(f"Test accuracy      : {loaded_info['test_accuracy']}")
print(f"Test F1            : {loaded_info['test_f1']}")

img_path     = test_paths[0]
image        = Image.open(img_path).convert("RGB")
words, boxes = get_ocr_data(img_path)  # reads from cache

enc = verify_processor(
    images=image,
    text=words,
    boxes=boxes,
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)

with torch.no_grad():
    out = verify_model(
        input_ids=enc["input_ids"].to(device),
        attention_mask=enc["attention_mask"].to(device),
        bbox=enc["bbox"].to(device),
        pixel_values=enc["pixel_values"].to(device),
    )

probs      = F.softmax(out.logits, dim=-1).squeeze(0)
pred_idx   = probs.argmax().item()
pred_class = loaded_info['class_names'][pred_idx]
confidence = probs[pred_idx].item() * 100

print(f"\nVerification prediction:")
print(f"  Image     : {Path(img_path).name}")
print(f"  Predicted : {pred_class}  ({confidence:.1f}%)")
print("\nModel verification successful!")


Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

Loaded class names : ['non_medical', 'prescription', 'report']
Architecture       : LayoutLMv3ForSequenceClassification
Test accuracy      : 0.9083
Test F1            : 0.9068

Verification prediction:
  Image     : non_medical_001.png
  Predicted : non_medical  (99.9%)

Model verification successful!
